I want to understand oauth 2.0 therefore I decided to implement in python from scratch understanding it step by step. For this exercise I referred the following excellent articles
- https://stack-auth.com/blog/oauth-from-first-principles
- https://developer.okta.com/blog/2019/10/21/illustrated-guide-to-oauth-and-oidc
- https://www.ducktyped.org/p/an-illustrated-guide-to-oauth

Also I took help of claude sonnet 4.6 wherever I can. I used solve.it.com for this exercise. 

Lets get started!

For starters oauth 2.0 allows application to access another application on your behalf. For example allowing an agentic application to  to access your gmail account to send automated replies on your behalf. 

Here we have AuthServer and Client App in the following architecture diagram. The client front end will be responsible for starting the flow and the subsequent requests will be maintained by the client backend with the generated token

```
Browser                  Auth Server          Client Backend
   |                         |                      |
   |--- /authorize --------->|                      |
   |<-- redirect(code) ------|                      |
   |                         |                      |
   |--- /callback(code) ---->|                      |
   |                         |<-- POST /token ------|
   |                         |--- access_token ---->|
   |                         |                      |
   |                         |             (future API calls)
   |                         |<-- Bearer token -----|
   |                         |--- protected data --->|
````

Let me define the APIs

In [ ]:
import httpx
from fasthtml.common import fast_app, Titled, P, FastHTML, RedirectResponse, Request
from fasthtml.jupyter import HTMX, JupyUvi
from functools import partial
import time
import fastcore.all as fc
from random import randint
import hashlib, base64

/usr/local/lib/python3.12/site-packages/fasthtml/jupyter.py:136: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient


In [ ]:
app = FastHTML()
rt = app.route

In [ ]:
def get_preview(app): return partial(HTMX, app=app, host=None, port=None)
p = get_preview(app)
srv = JupyUvi(app)

In [ ]:
@rt
def oauth_demo(): return Titled("OAuth Demo", P("Authorization Server - Step 1"))

In [ ]:
p(oauth_demo)

HTML(<iframe src="/oauth_demo" style="width: 100%; height: auto; border: none;" onload="{
        let frame = this;
        window.addEventListener('message', function(e) {
            if (e.source !== frame.contentWindow) return; // Only proceed if the message is from this iframe
            if (e.data.height) frame.style.height = (e.data.height+1) + 'px';
        }, false);
    }" allow="accelerometer; autoplay; camera; clipboard-read; clipboard-write; display-capture; encrypted-media; fullscreen; gamepad; geolocation; gyroscope; hid; identity-credentials-get; idle-detection; magnetometer; microphone; midi; payment; picture-in-picture; publickey-credentials-get; screen-wake-lock; serial; usb; web-share; xr-spatial-tracking"></iframe> )

In [ ]:
@rt
def oauth_demo_authorize(client_id: str, redirect_uri:str, state:int):
    return P(f"{client_id} is requesting access, redirecting to {redirect_uri}")